### TEST 02F · PVAR h1b con bootstrap por BLOQUES de choques (dominio pequeño)
Mismo PVAR h1b (`var_h1b.pkl`), solo cambia el ruido que se le pasa por fuera: en vez de choques sueltos, bloques de `L` choques consecutivos reales (mismo mes y misma hora de inicio). Así entra entera la secuencia de choques de una borrasca real.

Variantes `h1bb{L}`: L = 4, 8, 12 pasos (1, 2, 3 días). Todo en `outputs/var_6h_wmed/simulaciones_h1bb{L}/` y `outputs/grad_rec/simulaciones_h1bb{L}_wmed/`. Autocontenido: simula, reconstruye el gradiente y compara la f.

In [20]:
import os, glob, json, pickle, time

import pandas as pd
import numpy as np
import xarray as xr
from scipy.signal import find_peaks
from xgboost import XGBRegressor
from joblib import Parallel, delayed

from bluemath_tk.core.io import load_model
from bluemath_tk.core.operations import spatial_gradient

#### Load Data

In [21]:
out = 'outputs/var_6h_wmed'
var = pickle.load(open(f'{out}/var_h1b.pkl', 'rb'))
harm_std_pcs = pickle.load(open(f'{out}/harm_std_pcs.pkl', 'rb'))

burn = int(pd.Timedelta(days=3*365) / pd.Timedelta("6h"))
fechas = pd.date_range("1940-01-01", "2026-12-31 18:00", freq="6h")

BLOQUES = [24]
N_SIMS = 20

#### Ruido por bloques

In [22]:
R = var.resid_.values
idx = var.resid_.index
paso = pd.Timedelta('6h')

def inicios_validos(L):
    """Posiciones donde empieza un bloque de L residuos consecutivos, agrupadas por (mes, hora)."""
    pos = np.arange(len(idx) - L + 1)
    ok = (idx[pos + L - 1] - idx[pos]) == (L - 1) * paso
    pos = pos[ok]
    return {(m, h): pos[(idx.month[pos] == m) & (idx.hour[pos] == h)]
            for m in range(1, 13) for h in [0, 6, 12, 18]}

def ruido_bloques(L, fechas, seed, pools):
    rng = np.random.default_rng(seed)
    eps = np.empty((len(fechas), var.k_))
    for i0 in range(0, len(fechas), L):
        t = fechas[i0]
        pool = pools[(t.month, t.hour)]
        s = pool[rng.integers(len(pool))]
        n = min(L, len(fechas) - i0)
        eps[i0:i0 + n] = R[s:s + n]
    return pd.DataFrame(eps, index=fechas, columns=var.columns_)

#### Simulaciones

In [23]:
for L in BLOQUES:
    pools = inicios_validos(L)
    sim_dir = f'{out}/simulaciones_h1bb{L}'
    os.makedirs(sim_dir, exist_ok=True)
    for i in range(N_SIMS):
        f_sim = f'{sim_dir}/sim_{i:03d}.parquet'
        if os.path.exists(f_sim):
            continue
        t = time.time()
        sim_z = var.simulate(noise=ruido_bloques(L, fechas, seed=i, pools=pools), burn=burn)
        harm_std_pcs.inverse_transform(sim_z).astype('float32').to_parquet(f_sim)
        print(f'h1bb{L} sim {i:03d}  ({time.time() - t:.0f} s)')

h1bb24 sim 010  (7 s)
h1bb24 sim 011  (7 s)
h1bb24 sim 012  (7 s)
h1bb24 sim 013  (6 s)
h1bb24 sim 014  (7 s)
h1bb24 sim 015  (7 s)
h1bb24 sim 016  (7 s)
h1bb24 sim 017  (7 s)
h1bb24 sim 018  (7 s)
h1bb24 sim 019  (7 s)


#### Gradiente reconstruido (igual que el 03)

In [24]:
pca_msl = load_model('outputs/pca/pca_msl_wmed.pkl')
pca_grad = load_model('outputs/grad_rec/pca_grad_wmed.pkl')
pcs_msl = pd.read_csv('outputs/pca/pcs_msl_30_6h_wmed.csv', index_col=0, parse_dates=True)
cols = list(pcs_msl.columns)

def grad_pcs(pcs, n_gr=10, paso=8000):
    out = []
    for i in range(0, len(pcs), paso):
        p = pcs.iloc[i:i + paso][cols]
        pcs_da = xr.DataArray(p.values, dims=('time', 'n_component'),
                              coords={'time': p.index, 'n_component': np.arange(len(cols))})
        campo = pca_msl.inverse_transform(pcs_da)['msl'].transpose('time', 'latitude', 'longitude')
        grad = spatial_gradient(campo).to_dataset(name='grad2')
        out.append(pca_grad.transform(data=grad)['PCs'].values[:, :n_gr])
    return pd.DataFrame(np.concatenate(out), index=pcs.index, columns=[f'grec_PC{k + 1}' for k in range(n_gr)])

def procesar(f, out_dir):
    f_out = os.path.join(out_dir, os.path.basename(f).replace('sim_', 'grec_'))
    if not os.path.exists(f_out):
        grad_pcs(pd.read_parquet(f)).astype('float32').to_parquet(f_out)

for L in BLOQUES:
    sim_files = sorted(glob.glob(f'{out}/simulaciones_h1bb{L}/sim_*.parquet'))
    out_dir = f'outputs/grad_rec/simulaciones_h1bb{L}_wmed'
    os.makedirs(out_dir, exist_ok=True)
    print(f'h1bb{L}', len(sim_files))
    _ = Parallel(n_jobs=os.cpu_count() - 2, verbose=0)(delayed(procesar)(f, out_dir) for f in sim_files)

h1bb24

Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray a

 20


Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!


#### f: histórico vs h1b vs bloques

In [25]:
pcs_grec = pd.read_csv('outputs/grad_rec/pcs_grad_rec_6h_wmed.csv', index_col=0, parse_dates=True)
hist = pcs_msl.join(pcs_grec).loc['1941':'2025']

xgb_dir = 'outputs/xgb_p102_wmed_actual'
xgb = XGBRegressor()
xgb.load_model(f'{xgb_dir}/xgb_final.json')
meta = json.load(open(f'{xgb_dir}/xgb_features.json'))
lags = meta['lags_h']

def f_desde(d):
    Xs = pd.concat({f'l{l}': d.shift(freq=pd.Timedelta(hours=l)) for l in lags}, axis=1).reindex(d.index)
    Xs.columns = [f'{c}_l{l[1:]}' for l, c in Xs.columns]
    Xs = Xs.dropna()[meta['columns']]
    return pd.Series(xgb.predict(Xs), index=Xs.index)

def cargar_f(nombre, n=N_SIMS):
    fs = []
    for fm in sorted(glob.glob(f'{out}/simulaciones_{nombre}/sim_*.parquet'))[:n]:
        fg = os.path.join(f'outputs/grad_rec/simulaciones_{nombre}_wmed', os.path.basename(fm).replace('sim_', 'grec_'))
        sim = pd.read_parquet(fm).astype('float64')[cols].join(pd.read_parquet(fg).astype('float64')).loc['1941':'2025']
        fs.append(f_desde(sim))
    return fs

f_hist = f_desde(hist)
umbral = f_hist.quantile(0.99)
f_sims = {n: cargar_f(n) for n in ['h1b'] + [f'h1bb{L}' for L in BLOQUES]}

In [26]:
def resumen(f):
    pk = find_peaks(f.values, height=umbral, distance=12)[0]
    return {'q90': f.quantile(0.90), 'q99': f.quantile(0.99), 'q99.9': f.quantile(0.999),
            'máx anual medio': f.groupby(f.index.year).max().mean(),
            'picos f > q99 hist / década': len(pk) * 10 / f.index.year.nunique()}

def episodios(f, u=umbral):
    sobre = f > u
    g = (sobre != sobre.shift()).cumsum()[sobre]
    dur = sobre[sobre].groupby(g).size() * 6
    n_dec = f.index.year.nunique() / 10
    return {'% tiempo > q99': sobre.mean() * 100, 'episodios / década': len(dur) / n_dec,
            'duración media [h]': dur.mean(), 'duración p90 [h]': dur.quantile(0.9)}

h = pd.Series(resumen(f_hist))
display(pd.DataFrame({f'{n} / hist': pd.DataFrame([resumen(f) for f in fs]).median() / h for n, fs in f_sims.items()}).round(3))
pd.DataFrame({'histórico': episodios(f_hist), **{n: pd.DataFrame([episodios(f) for f in fs]).median() for n, fs in f_sims.items()}}).round(2)

,h1b / hist,h1bb24 / hist
q90,0.906,0.970
q99,0.967,0.990
q99.9,1.007,0.996
máx anual medio,0.995,0.990
picos f > q99 hist / década,0.794,0.902


,histórico,h1b,h1bb24
% tiempo > q99,1.00,0.85,0.95
episodios / década,52.00,41.82,47.53
duración media [h],16.86,17.73,17.65
duración p90 [h],30.00,36.00,36.00


In [27]:
est = lambda m: np.select([np.isin(m, [12, 1, 2]), np.isin(m, [3, 4, 5]), np.isin(m, [6, 7, 8])], ['invierno', 'primavera', 'verano'], 'otoño')

def picos_est(f):
    pk = f.index[find_peaks(f.values, height=umbral, distance=12)[0]]
    return pd.Series(est(pk.month)).value_counts() * 10 / f.index.year.nunique()

pd.DataFrame({'histórico': picos_est(f_hist),
              **{n: pd.concat([picos_est(f) for f in fs], axis=1).fillna(0).median(axis=1) for n, fs in f_sims.items()}}
            ).loc[['invierno', 'primavera', 'verano', 'otoño']].round(1)

,histórico,h1b,h1bb24
invierno,21.2,17.3,19.2
primavera,12.9,7.4,10.9
verano,4.2,5.7,4.8
otoño,10.7,8.1,9.3


#### Control: σ y ACF de las PCs (que los bloques no rompan el 2º orden)

In [28]:
ref = pcs_msl.loc['1941':'2025']
filas = {}
for n in ['h1b'] + [f'h1bb{L}' for L in BLOQUES]:
    sims = [pd.read_parquet(f).astype('float64').loc['1941':'2025'] for f in sorted(glob.glob(f'{out}/simulaciones_{n}/sim_*.parquet'))[:5]]
    filas[n] = {'σ sim/hist (mediana PCs)': np.median(pd.concat([s.std() for s in sims], axis=1).median(axis=1) / ref.std()),
                'ACF 1 d sim − hist (mediana PCs)': np.median(np.median([[s[c].autocorr(4) for c in cols] for s in sims], axis=0)
                                                              - np.array([ref[c].autocorr(4) for c in cols]))}
pd.DataFrame(filas).round(3)

,h1b,h1bb24
σ sim/hist (mediana PCs),1.001,1.002
ACF 1 d sim − hist (mediana PCs),0.002,0.004


In [29]:
#### 1. ¿Hay agrupamiento de volatilidad en los choques? ACF de la norma² del choque
n2 = pd.Series((R ** 2).sum(axis=1), index=idx)
print({f'{6 * l} h': round(n2.autocorr(l), 3) for l in [1, 2, 4, 8, 12]})

#### 2. Ruido "solo volatilidad": tamaño del choque en bloques reales, dirección de choques sueltos al azar
norma = np.linalg.norm(R, axis=1)

def ruido_vol(L, fechas, seed, pools):
    rng = np.random.default_rng(seed)
    eps = np.empty((len(fechas), var.k_))
    for i0 in range(0, len(fechas), L):
        t = fechas[i0]
        pool = pools[(t.month, t.hour)]
        s = pool[rng.integers(len(pool))]
        n = min(L, len(fechas) - i0)
        d = var.resid_month_[t.month][rng.integers(len(var.resid_month_[t.month]), size=n)]
        eps[i0:i0 + n] = d / np.linalg.norm(d, axis=1, keepdims=True) * norma[s:s + n, None]
    return pd.DataFrame(eps, index=fechas, columns=var.columns_)

L = 24
pools = inicios_validos(L)
sim_dir = f'{out}/simulaciones_h1bv{L}'
os.makedirs(sim_dir, exist_ok=True)
for i in range(N_SIMS):
    f_sim = f'{sim_dir}/sim_{i:03d}.parquet'
    if not os.path.exists(f_sim):
        sim_z = var.simulate(noise=ruido_vol(L, fechas, seed=i, pools=pools), burn=burn)
        harm_std_pcs.inverse_transform(sim_z).astype('float32').to_parquet(f_sim)

out_dir = f'outputs/grad_rec/simulaciones_h1bv{L}_wmed'
os.makedirs(out_dir, exist_ok=True)
_ = Parallel(n_jobs=os.cpu_count() - 2)(delayed(procesar)(f, out_dir) for f in sorted(glob.glob(f'{sim_dir}/sim_*.parquet')))

f_sims[f'h1bv{L}'] = cargar_f(f'h1bv{L}')
display(pd.DataFrame({f'{n} / hist': pd.DataFrame([resumen(f) for f in fs]).median() / h for n, fs in f_sims.items()}).round(3))
pd.DataFrame({'histórico': picos_est(f_hist),
              **{n: pd.concat([picos_est(f) for f in fs], axis=1).fillna(0).median(axis=1) for n, fs in f_sims.items()}}
            ).loc[['invierno', 'primavera', 'verano', 'otoño']].round(1)

{'6 h': np.float64(0.639), '12 h': np.float64(0.501), '24 h': np.float64(0.334), '48 h': np.float64(0.154), '72 h': np.float64(0.093)}


Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray and will be pickled!
Attribute pcs is an xarray Dataset / Dataarray a

,h1b / hist,h1bb24 / hist,h1bv24 / hist
q90,0.906,0.970,0.927
q99,0.967,0.990,0.965
q99.9,1.007,0.996,0.996
máx anual medio,0.995,0.990,0.982
picos f > q99 hist / década,0.794,0.902,0.742


,histórico,h1b,h1bb24,h1bv24
invierno,21.2,17.3,19.2,14.6
primavera,12.9,7.4,10.9,8.2
verano,4.2,5.7,4.8,6.4
otoño,10.7,8.1,9.3,7.9


In [30]:
#### Banda de confianza: picos de f por década en bloques de 41 años (como el 09B)
def picos_dec(f):
    return len(find_peaks(f.values, height=umbral, distance=12)[0]) * 10 / f.index.year.nunique()

def bloques41(f):
    a0, a1 = f.index.year.min(), f.index.year.max()
    return [f.loc[str(a):str(a + 40)] for a in range(a0, a1 - 39, 41)]

obs = picos_dec(f_hist.loc['1985':'2025'])                       # mismo periodo que el oleaje observado
filas = {}
for n, fs in f_sims.items():
    v = np.array([picos_dec(b) for f in fs for b in bloques41(f)])
    filas[n] = {'n bloques': len(v), 'p5': np.percentile(v, 5), 'p50': np.median(v), 'p95': np.percentile(v, 95),
                'rango percentil obs': (v < obs).mean() * 100}
print(f'observado 1985–2025: {obs:.1f} picos/década  (1941–2025: {picos_dec(f_hist):.1f})')
pd.DataFrame(filas).T.round(1)

observado 1985–2025: 49.5 picos/década  (1941–2025: 49.1)


,n bloques,p5,p50,p95,rango percentil obs
h1b,40.0,34.3,38.8,41.5,100.0
h1bb24,40.0,39.2,43.4,49.3,95.0
h1bv24,40.0,31.4,37.1,42.9,100.0
